In [ ]:
import torch
import torch.nn as nn
import math
from transformers.activations import gelu


# -------------------------------------------------
# Embeddings
# -------------------------------------------------

class BertEmbeddings(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.position_embeddings = nn.Embedding(
            config.max_position_embeddings,
            config.hidden_size
        )

        self.LayerNorm = nn.LayerNorm(
            config.hidden_size,
            eps=config.layer_norm_eps
        )
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def embedding_single(self, input_embeds):

        batch_size, seq_length, _ = input_embeds.size()
        device = input_embeds.device

        # Position ids
        position_ids = torch.arange(seq_length, device=device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_length)

        # Position + token embeddings
        positions = self.position_embeddings(position_ids)

        embeddings = input_embeds + positions
        embeddings = self.LayerNorm(embeddings)
        embeddings = self.dropout(embeddings)

        return embeddings

    def forward(self, embed1, embed2, embed3):

        out1 = self.embedding_single(embed1)
        out2 = self.embedding_single(embed2)
        out3 = self.embedding_single(embed3)

        return out1, out2, out3
    
# -------------------------------------------------
# Projection for Residual Stream
# -------------------------------------------------
    
class TripleSubspaceProjection(nn.Module):
    def __init__(self, input_dim=768, subspace_dim=256):
        super().__init__()

        # Three independent projection matrices
        self.proj1 = nn.Linear(input_dim, subspace_dim, bias=False)
        self.proj2 = nn.Linear(input_dim, subspace_dim, bias=False)
        self.proj3 = nn.Linear(input_dim, subspace_dim, bias=False)

    def forward(self, embed1, embed2, embed3):
        """
        embedX: (batch_size, seq_length, 768)
        returns:
            projX: (batch_size, seq_length, 256)
        """

        out1 = self.proj1(embed1)
        out2 = self.proj2(embed2)
        out3 = self.proj3(embed3)

        combined = torch.cat([out1, out2, out3], dim=-1)

        return combined
    
# -------------------------------------------------
# Self Attention
# -------------------------------------------------

class BertSelfAttention0(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.all_head_size = self.num_heads * self.head_dim

        self.query = nn.ModuleList([
            nn.Linear(config.hidden_size, self.head_dim)
            for _ in range(self.num_heads)
        ])
        self.key = nn.ModuleList([
            nn.Linear(config.hidden_size, self.head_dim)
            for _ in range(self.num_heads)
        ])
        self.value = nn.ModuleList([
            nn.Linear(config.hidden_size, self.head_dim)
            for _ in range(self.num_heads)
        ])

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

    def forward(self, embeds1, embeds2, embeds3):  

        head_outputs = []

        for h in range(self.num_heads):

            inputs = 0
            if h < 4:
                inputs = embeds1
            elif h < 8:
                inputs = embeds2
            elif h < 12:
                inputs = embeds3 

            Q = self.query[h](inputs)  
            K = self.key[h](inputs)   
            V = self.value[h](inputs)

            scores = torch.matmul(Q, K.transpose(-1, -2))
            scores = scores / math.sqrt(self.head_dim)

            probs = torch.softmax(scores, dim=-1)
            probs = self.dropout(probs)

            context = torch.matmul(probs, V)

            head_outputs.append(context)

        context = torch.cat(head_outputs, dim=-1)

        return context

class BertSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.all_head_size = self.num_heads * self.head_dim
        block_dim = config.hidden_size // 3
        num_modality_heads = config.num_attention_heads // 3

        self.query1 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])
        self.key1 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])
        self.value1 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])

        self.query2 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])
        self.key2 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])
        self.value2 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])

        self.query3 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])
        self.key3 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])
        self.value3 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

    def forward(self, inputs):  

        block_dim = inputs.size(-1) // 3

        x1 = inputs[..., :block_dim]
        x2 = inputs[..., block_dim:2*block_dim]
        x3 = inputs[..., 2*block_dim:]

        head_outputs = []

        # ---- Modality 1 ----
        for h in range(len(self.query1)):
            Q = self.query1[h](x1)
            K = self.key1[h](x1)
            V = self.value1[h](x1)

            scores = torch.matmul(Q, K.transpose(-1, -2))
            scores = scores / math.sqrt(self.head_dim)
            probs = torch.softmax(scores, dim=-1)
            probs = self.dropout(probs)

            context = torch.matmul(probs, V)
            head_outputs.append(context)

        # ---- Modality 2 ----
        for h in range(len(self.query2)):
            Q = self.query2[h](x2)
            K = self.key2[h](x2)
            V = self.value2[h](x2)

            scores = torch.matmul(Q, K.transpose(-1, -2))
            scores = scores / math.sqrt(self.head_dim)
            probs = torch.softmax(scores, dim=-1)
            probs = self.dropout(probs)

            context = torch.matmul(probs, V)
            head_outputs.append(context)

        # ---- Modality 3 ----
        for h in range(len(self.query3)):
            Q = self.query3[h](x3)
            K = self.key3[h](x3)
            V = self.value3[h](x3)

            scores = torch.matmul(Q, K.transpose(-1, -2))
            scores = scores / math.sqrt(self.head_dim)
            probs = torch.softmax(scores, dim=-1)
            probs = self.dropout(probs)

            context = torch.matmul(probs, V)
            head_outputs.append(context)

        context = torch.cat(head_outputs, dim=-1)

        return context
    
class BertCrossAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.all_head_size = self.num_heads * self.head_dim
        block_dim = config.hidden_size // 3
        num_modality_heads = config.num_attention_heads // 3

        self.query1 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])
        self.key1 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])
        self.value1 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])

        self.query2 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])
        self.key2 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])
        self.value2 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])

        self.query3 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])
        self.key3 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])
        self.value3 = nn.ModuleList([
            nn.Linear(block_dim, self.head_dim)
            for _ in range(num_modality_heads)
        ])

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

    def attend(self, Q, K, V):
        scores = torch.matmul(Q, K.transpose(-1, -2))
        scores = scores / math.sqrt(self.head_dim)
        probs = torch.softmax(scores, dim=-1)
        probs = self.dropout(probs)
        return torch.matmul(probs, V)

    def forward(self, inputs):  

        block_dim = inputs.size(-1) // 3

        x1 = inputs[..., :block_dim]
        x2 = inputs[..., block_dim:2*block_dim]
        x3 = inputs[..., 2*block_dim:]

        head_outputs = []

        for h in range(2):
            Q = self.query1[h](x1)
            K = self.key2[h](x2)
            V = self.value2[h](x2)
            head_outputs.append(self.attend(Q, K, V))

        for h in range(2, 4):
            Q = self.query1[h](x1)
            K = self.key3[h-2](x3)
            V = self.value3[h-2](x3)
            head_outputs.append(self.attend(Q, K, V))

        # ---- Modality 2 ----
        for h in range(2):
            Q = self.query2[h](x2)
            K = self.key1[h](x1)
            V = self.value1[h](x1)
            head_outputs.append(self.attend(Q, K, V))

        for h in range(2, 4):
            Q = self.query2[h](x2)
            K = self.key3[h](x3)
            V = self.value3[h](x3)
            head_outputs.append(self.attend(Q, K, V))

        # ---- Modality 3 ----
        for h in range(2):
            Q = self.query3[h](x3)
            K = self.key1[h+2](x1)
            V = self.value1[h+2](x1)
            head_outputs.append(self.attend(Q, K, V))

        for h in range(2, 4):
            Q = self.query3[h](x3)
            K = self.key2[h](x2)
            V = self.value2[h](x2)
            head_outputs.append(self.attend(Q, K, V))

        return torch.cat(head_outputs, dim=-1)

class BlockDiagonalLinear(nn.Module):
    def __init__(self, in_size, out_size, block_size=256, bias=True):
        super().__init__()

        self.block_size_in = block_size
        self.num_blocks = in_size // block_size
        self.block_size_out = out_size // self.num_blocks

        self.weight = nn.Parameter(
            torch.randn(self.num_blocks, block_size, self.block_size_out)
        )
        
        if bias:
            self.bias = nn.Parameter(
                torch.zeros(self.num_blocks, self.block_size_out)
            )
        else:
            self.bias = None

    def forward(self, x):

        original_shape = x.shape
        hidden_size = original_shape[-1]

        # Flatten everything except hidden dimension
        x = x.view(-1, hidden_size)

        # Reshape into blocks
        x = x.view(-1, self.num_blocks, self.block_size_in)

        # Block-wise matmul
        out = torch.einsum('bnd,ndh->bnh', x, self.weight)

        if self.bias is not None:
            out = out + self.bias

        # Flatten blocks back
        out = out.reshape(out.shape[0], -1)

        # Restore original dimensions
        new_hidden = out.shape[-1]
        out = out.view(*original_shape[:-1], new_hidden)

        return out
    
class BlockLayerNorm(nn.Module):
    def __init__(self, hidden_size, block_size=256, eps=1e-12):
        super().__init__()
        
        self.block_size = block_size
        self.num_blocks = hidden_size // block_size
        self.eps = eps
        
        # Separate gamma and beta per block
        self.weight = nn.Parameter(
            torch.ones(self.num_blocks, block_size)
        )
        self.bias = nn.Parameter(
            torch.zeros(self.num_blocks, block_size)
        )

    def forward(self, x):
        # x: (batch, seq_len, hidden_size) OR (batch, hidden_size)
        original_shape = x.shape
        
        # Merge all dims except last
        x = x.view(-1, self.num_blocks, self.block_size)
        
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, unbiased=False, keepdim=True)
        
        x = (x - mean) / torch.sqrt(var + self.eps)
        
        x = x * self.weight + self.bias
        
        return x.view(original_shape)

class BertSelfOutput(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = BlockDiagonalLinear(config.hidden_size, config.hidden_size, block_size=256)
        self.LayerNorm = BlockLayerNorm(
            config.hidden_size,
            block_size=256,
            eps=config.layer_norm_eps
        )
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states

# -------------------------------------------------
# Feed Forward
# -------------------------------------------------

class BertIntermediate(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = BlockDiagonalLinear(config.hidden_size, config.intermediate_size, block_size=256)

    def forward(self, hidden_states):
        return gelu(self.dense(hidden_states))


class BertOutput(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = BlockDiagonalLinear(config.intermediate_size, config.hidden_size, block_size=256)
        self.LayerNorm = BlockLayerNorm(
            config.hidden_size,
            block_size=256,
            eps=config.layer_norm_eps
        )
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states


# -------------------------------------------------
# Encoder Layer
# -------------------------------------------------

class BertLayer0(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention = nn.ModuleDict({
            "self": BertSelfAttention0(config),
            "output": BertSelfOutput(config)
        })
        self.intermediate = BertIntermediate(config)
        self.output = BertOutput(config)

    def forward(self, input_embeds1, input_embeds2, input_embeds3, residual_stream):
        attn_output = self.attention["self"](input_embeds1, input_embeds2, input_embeds3)
        hidden_states = self.attention["output"](attn_output, residual_stream)

        intermediate_output = self.intermediate(hidden_states)
        layer_output = self.output(intermediate_output, hidden_states)

        return layer_output

class BertLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention = nn.ModuleDict({
            "self": BertSelfAttention(config),
            "output": BertSelfOutput(config)
        })
        self.intermediate = BertIntermediate(config)
        self.output = BertOutput(config)

    def forward(self, hidden_states):
        attn_output = self.attention["self"](hidden_states)
        hidden_states = self.attention["output"](attn_output, hidden_states)

        intermediate_output = self.intermediate(hidden_states)
        layer_output = self.output(intermediate_output, hidden_states)

        return layer_output
    
class BertXLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention = nn.ModuleDict({
            "cross": BertCrossAttention(config),
            "output": BertSelfOutput(config)
        })
        self.intermediate = BertIntermediate(config)
        self.output = BertOutput(config)

    def forward(self, hidden_states):
        attn_output = self.attention["cross"](hidden_states)
        hidden_states = self.attention["output"](attn_output, hidden_states)

        intermediate_output = self.intermediate(hidden_states)
        layer_output = self.output(intermediate_output, hidden_states)

        return layer_output

# -------------------------------------------------
# Full BERT
# -------------------------------------------------

class CustomBertModel(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.embeddings = BertEmbeddings(config)
        self.subspace_proj = TripleSubspaceProjection()

        self.encoder = nn.ModuleDict({
            "layer": nn.ModuleList([
                BertLayer0(config) if i == 0
                else BertLayer(config) if i <= 4
                else BertXLayer(config)
                for i in range(config.num_hidden_layers)
            ])
        })

        self.pooler = nn.ModuleDict({
            "dense": BlockDiagonalLinear(config.hidden_size, config.hidden_size, block_size=256)
        })
        self.pooler_activation = nn.Tanh()

    def forward(self, input_embeds1, input_embeds2, input_embeds3):

        input_embeds1, input_embeds2, input_embeds3 = self.embeddings(input_embeds1, input_embeds2, input_embeds3)
        residual_stream = self.subspace_proj(input_embeds1, input_embeds2, input_embeds3)

        hidden_states = self.encoder["layer"][0](input_embeds1, input_embeds2, input_embeds3, residual_stream)

        for layer_module in self.encoder["layer"][1:]:
            hidden_states = layer_module(hidden_states)

        cls_token = hidden_states.mean(dim=1)
        pooled_output = self.pooler_activation(
            self.pooler["dense"](cls_token)
        )

        return hidden_states, pooled_output

In [15]:
from transformers import BertModel, BertConfig
import torch

hf_model = BertModel.from_pretrained("bert-base-uncased")
hf_model.eval()

config = hf_model.config

my_model = CustomBertModel(config)
my_model.eval()

CustomBertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): ModuleDict(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): ModuleDict(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
   

In [16]:
hf_state_dict = hf_model.state_dict()
my_state_dict = my_model.state_dict()

# Check missing/unexpected keys
missing = []
for name in my_state_dict:
    if name not in hf_state_dict:
        missing.append(name)

print("Missing keys:", missing)

Missing keys: []


In [18]:
my_model.load_state_dict(hf_state_dict, strict=True)

<All keys matched successfully>

In [19]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

text = "Hello world, this is a test."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    hf_outputs = hf_model(**inputs)
    my_outputs = my_model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        token_type_ids=inputs["token_type_ids"]
    )

In [20]:
hf_last = hf_outputs.last_hidden_state
hf_pool = hf_outputs.pooler_output

my_last, my_pool = my_outputs

print("Hidden difference:",
      torch.max(torch.abs(hf_last - my_last)))

print("Pool difference:",
      torch.max(torch.abs(hf_pool - my_pool)))

Hidden difference: tensor(1.9073e-06)
Pool difference: tensor(5.5134e-07)


In [21]:
torch.allclose(hf_last, my_last, atol=1e-6)

True